# 👕 Virtual Try-On — Google Colab Setup

**Bilkul free. Koi payment nahi. Bas cells upar se neeche run karo.**

---

### Pehle Yeh Karo (Ek Baar):
1. Menu → **Runtime** → **Change runtime type**
2. Hardware accelerator = **T4 GPU** select karo
3. **Save** karo
4. Phir neeche cells run karo

---
| Cell | Kya karta hai | Time |
|------|--------------|------|
| 1 | GPU check | 5 sec |
| 2 | PyTorch + CUDA verify | 10 sec |
| 3 | Detectron2 install | 3-5 min |
| 4 | Project clone | 1-2 min |
| 5 | Requirements install | 3-5 min |
| 6 | DensePose model download | 1-2 min |
| 7 | App launch (public URL milega) | 15-20 min (model load) |

In [ ]:
# ── Cell 1: GPU Check ─────────────────────────────────────────────────────────
import torch

print('='*50)
print('GPU Available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU Name   :', torch.cuda.get_device_name(0))
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print('VRAM       :', vram_gb, 'GB')
    print('CUDA Ver   :', torch.version.cuda)
    print('PyTorch Ver:', torch.__version__)
    print('='*50)
    if vram_gb < 12:
        print('⚠️  WARNING: 12GB+ VRAM chahiye. T4 (15GB) use karo.')
    else:
        print('✅ GPU ready hai!')
else:
    print('='*50)
    print('❌ GPU nahi mila!')
    print('   Runtime > Change runtime type > T4 GPU select karo')
    raise SystemExit('GPU required — runtime type change karo phir restart karo')

In [ ]:
# ── Cell 2: Detectron2 Install (special — pip se direct nahi hota) ────────────
# Yeh automatically sahi version detect kar ke install karta hai

import torch
import subprocess

cuda_ver = torch.version.cuda.replace('.', '')[:3]   # e.g. '121'
torch_ver = torch.__version__.split('+')[0]           # e.g. '2.3.0'
torch_major_minor = 'torch' + '.'.join(torch_ver.split('.')[:2])  # e.g. 'torch2.3'

wheel_url = f'https://dl.fbaipublicfiles.com/detectron2/wheels/cu{cuda_ver}/{torch_major_minor}/index.html'

print(f'CUDA: {cuda_ver} | PyTorch: {torch_ver}')
print(f'Wheel URL: {wheel_url}')
print('Detectron2 install ho raha hai...')

# Pehle pre-built wheel try karo (fast)
result = subprocess.run(
    ['pip', 'install', 'detectron2', '-f', wheel_url, '-q'],
    capture_output=True, text=True
)

if result.returncode != 0:
    # Pre-built wheel nahi mila — source se build karo (slow but works)
    print('Pre-built wheel nahi mila. Source se build ho raha hai (5-10 min)...')
    subprocess.run(
        ['pip', 'install', 'git+https://github.com/facebookresearch/detectron2.git', '-q'],
        check=True
    )

# Verify
import detectron2
print(f'✅ Detectron2 {detectron2.__version__} install ho gaya!')

In [ ]:
# ── Cell 3: Project Clone from GitHub ────────────────────────────────────────
import os, subprocess

GITHUB_USERNAME = 'zaryabali786'
REPO_NAME       = 'try-your-self'
PROJECT_DIR     = '/content/try-your-self'

GITHUB_URL = f'https://github.com/{GITHUB_USERNAME}/{REPO_NAME}'

os.chdir('/content')

if os.path.isdir(PROJECT_DIR):
    print(f'✅ Project already hai: {PROJECT_DIR}')
else:
    print(f'Cloning from: {GITHUB_URL}')
    result = subprocess.run(['git', 'clone', GITHUB_URL, PROJECT_DIR])
    if result.returncode == 0:
        print('✅ Clone complete!')
    else:
        raise RuntimeError('❌ Clone fail hua. GitHub repo public hai?')

os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
print('Files:', os.listdir('.'))

In [ ]:
# ── Cell 4: Requirements Install ─────────────────────────────────────────────

print('[1/6] diffusers aur gradio remove kar raha hai...')
!pip uninstall -y diffusers gradio 2>/dev/null; echo "done"

print()
print('[2/6] diffusers 0.20.2 install (--no-deps)...')
!pip install -q --no-deps "diffusers==0.20.2"
!pip install -q "safetensors>=0.3.1" "omegaconf"

print()
print('[3/6] huggingface_hub 1.x compatibility patches...')
import os, glob, sys

# ── File patch: constants.py (hf_cache_home) ─────────────────────
for path in glob.glob('/usr/local/lib/python*/dist-packages/diffusers/utils/constants.py'):
    with open(path) as f: src = f.read()
    OLD = 'from huggingface_hub.constants import HUGGINGFACE_HUB_CACHE, hf_cache_home'
    NEW = ('from huggingface_hub.constants import HUGGINGFACE_HUB_CACHE\n'
           'hf_cache_home = os.path.expanduser(\n'
           '    os.getenv("HF_HOME", os.path.join(os.getenv("XDG_CACHE_HOME","~/.cache"),"huggingface"))\n'
           ')')
    if OLD in src:
        with open(path, 'w') as f: f.write(src.replace(OLD, NEW))
        print(f'    ✅ constants.py patched')
    else:
        print(f'    ℹ️  constants.py already ok')

# ── In-memory shim: HfFolder + cached_download ───────────────────
# These were removed in huggingface_hub 1.0 — add them back
import huggingface_hub as _hf

if not hasattr(_hf, 'HfFolder'):
    class _HfFolder:
        @staticmethod
        def get_token():
            try: return _hf.get_token()
            except: return None
        @staticmethod
        def save_token(token):
            try: _hf.login(token=token, add_to_git_credential=False)
            except: pass
        @staticmethod
        def delete_token(): pass
    _hf.HfFolder = _HfFolder
    print('    ✅ HfFolder shim added')

if not hasattr(_hf, 'cached_download'):
    def _cached_download(*args, **kwargs):
        try: return _hf.hf_hub_download(*args, **kwargs)
        except: return args[0] if args else None
    _hf.cached_download = _cached_download
    print('    ✅ cached_download shim added')

# ── Verify diffusers import ───────────────────────────────────────
for mod in [k for k in sys.modules if 'diffusers' in k]:
    del sys.modules[mod]

import diffusers
print(f'    diffusers: {diffusers.__version__}')
from diffusers.models.embeddings import PositionNet
print(f'    PositionNet: OK ✅')

print()
print('[4/6] gradio 3.41.2...')
!pip install -q "gradio==3.41.2"

print()
print('[5/6] Image + model libraries...')
!pip install -q Pillow numpy scipy scikit-image opencv-python
!pip install -q einops onnxruntime cloudpickle

print()
print('[6/6] Detectron2 deps + API server...')
!pip install -q fvcore iopath pycocotools termcolor tabulate psutil
!pip install -q matplotlib tqdm basicsr
!pip install -q fastapi "uvicorn[standard]" python-multipart

print()
print('✅ Sab ho gaya!')

In [ ]:
# ── Cell 5: DensePose Model Download ─────────────────────────────────────────
import os, subprocess

os.chdir('/content/try-your-self')

ckpt_dir  = '/content/try-your-self/ckpt/densepose'
ckpt_file = os.path.join(ckpt_dir, 'model_final_162be9.pkl')

if os.path.exists(ckpt_file):
    size_mb = os.path.getsize(ckpt_file) / 1e6
    print(f'✅ DensePose model already hai ({size_mb:.0f} MB) — skip')
else:
    os.makedirs(ckpt_dir, exist_ok=True)
    print('DensePose model download ho raha hai (~250 MB)...')
    subprocess.run([
        'wget', '-q', '--show-progress',
        'https://dl.fbaipublicfiles.com/detectron2/densepose/densepose_rcnn_R_50_FPN_s1x/165712039/model_final_162be9.pkl',
        '-O', ckpt_file
    ], check=True)
    print(f'✅ Download complete!')

In [ ]:
# ── Cell 6: Final Verify ─────────────────────────────────────────────────────
import os

os.chdir('/content/try-your-self')

checks = {
    'app.py'                                   : os.path.exists('app.py'),
    'api.py'                                   : os.path.exists('api.py'),
    'configs/densepose_rcnn_R_50_FPN_s1x.yaml' : os.path.exists('configs/densepose_rcnn_R_50_FPN_s1x.yaml'),
    'ckpt/densepose/model_final_162be9.pkl'    : os.path.exists('ckpt/densepose/model_final_162be9.pkl'),
    'src/tryon_pipeline.py'                    : os.path.exists('src/tryon_pipeline.py'),
    'preprocess/'                              : os.path.exists('preprocess'),
}

all_ok = True
print('=== Pre-flight Check ===')
for name, ok in checks.items():
    status = '✅' if ok else '❌'
    print(f'  {status}  {name}')
    if not ok:
        all_ok = False

print()
if all_ok:
    print('✅ Sab ready hai! Cell 7 run karo.')
else:
    print('❌ Kuch missing hai — upar ke cells dobara check karo.')

---
## App Launch Karo

**Cell 7** = Gradio Web UI (browser mein use karo)  
**Cell 8** = FastAPI (Postman / mobile app se test karo)  

Dono mein se ek hi chalao — dono ek saath nahi.

In [ ]:
# ── Cell 7: Gradio Web UI Launch ──────────────────────────────────────────────
import os
os.chdir('/content/try-your-self')

print('App launch ho rahi hai...')
print('"Running on public URL: https://..." aa ne tak wait karo')
print()

!python app.py

In [ ]:
# ── Cell 8: FastAPI + ngrok Public URL ───────────────────────────────────────
import os, threading, time
os.chdir('/content/try-your-self')

!pip install -q pyngrok

from pyngrok import ngrok, conf

# ngrok auth token
ngrok.set_auth_token("cr_3CwBSXiGbcKrxhiM4DrnLMbqkmN")

# API server background mein start karo
def start_api():
    os.system('cd /content/try-your-self && python api.py')

thread = threading.Thread(target=start_api, daemon=True)
thread.start()

print('API server start ho raha hai...')
time.sleep(10)

# Public tunnel
public_url = ngrok.connect(8000)
print()
print('='*60)
print(f'✅ API Public URL: {public_url}')
print('='*60)
print()
print('Postman mein use karo:')
print(f'  GET  {public_url}/health')
print(f'  POST {public_url}/tryon')
print(f'  POST {public_url}/tryon/both')
print()
print('Swagger docs: ' + str(public_url) + '/docs')

---
## Masail / Troubleshooting

| Masla | Hall |
|-------|------|
| `CUDA out of memory` | Runtime > Restart, phir Cell 7 directly run karo |
| `No module named detectron2` | Cell 2 dobara run karo |
| `model_final_162be9.pkl not found` | Cell 5 dobara run karo |
| URL nahi aa raha | 2-3 min wait karo, Cell 7 ki output scroll karo |
| Colab disconnect ho gaya | Sab cells fresh run karo (model cache rehta hai Drive mein nahi, phir download hoga) |

---
**Note:** Colab free tier mein session ~12 ghante chalta hai. Baad mein phir se run karna parega.